# EEG-tGAT: Temporal Graph Attention Framework for EEG Classification

## 1. Introduction

This notebook implements the EEG-tGAT framework, a spatiotemporal graph attention model designed for EEG-based binary classification.

The framework integrates:
- Temporal convolutional feature extraction
- Channel-wise graph construction
- Graph Attention Network (GAT) layers
- End-to-end supervised learning

The objective is to model both intra-channel temporal dynamics and inter-channel spatial dependencies for robust EEG representation learning.


## 2. Experimental Setup

### Dataset
Briefly describe:
- Number of subjects
- Task type
- Binary labels meaning

### Preprocessing
- Channel selection
- Filtering
- ICA usage (if enabled)
- Trial segmentation strategy

In [ ]:
torch>=2.0.0
torch-geometric>=2.4.0
mne>=1.5.0
numpy>=1.23.0
scipy>=1.10.0
scikit-learn>=1.3.0
networkx>=3.0
ts2vg>=1.2.0
matplotlib>=3.7.0
seaborn>=0.12.0
pandas>=1.5.0

In [ ]:
import os
import numpy as np
import mne
import torch
import tempfile
import shutil
from mne.preprocessing import ICA, create_eog_epochs
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool, GATConv, GraphNorm, GATv2Conv
from torch_geometric.loader import DataLoader
import math
import networkx as nx
from scipy.signal import welch
from glob import glob
from ts2vg import NaturalVG
from sklearn.model_selection import GroupKFold, KFold
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix
)

In [ ]:
USE_ICA=False

SRC_DIR="/content/eeg-dataset"
DST_DIR="/content/eeg_clean"
os.makedirs(DST_DIR,exist_ok=True)

VALID_BIN={"Comment/mi_start":1,"Comment/Cg_start":0}
EXCLUDE={"Comment/mi_stop","Comment/AO_start","Comment/Cf_start"}

CHANNELS=['Fp1','Fp2','F3','F4','Fz','C3','C4','Cz','P3','P4','Pz','O1','O2','TP9','TP10','VEOG']

def prepare_dataset():
    for vhdr in glob(os.path.join(SRC_DIR,"*.ahdr")):
        base=os.path.splitext(os.path.basename(vhdr))[0]
        for ext in [".ahdr",".eeg",".amrk"]:
            src=os.path.join(SRC_DIR,base+ext)
            dst=os.path.join(DST_DIR,base+ext)
            if not os.path.exists(dst): shutil.copy(src,dst)
        with open(os.path.join(DST_DIR,base+".ahdr"),"r",encoding="utf-8",errors="ignore") as f:
            lines=f.readlines()
        clean=[]; skip=False
        for l in lines:
            if "Impedance" in l: skip=True; continue
            if skip and l.strip()=="":
                skip=False; continue
            if skip: continue
            clean.append(l)
        with open(os.path.join(DST_DIR,base+".ahdr"),"w",encoding="utf-8") as f:
            f.writelines(clean)

def load_raw(vhdr_path):
    raw=mne.io.read_raw_brainvision(vhdr_path,preload=True,eog=("VEOG",),verbose=False)
    raw._data=np.nan_to_num(raw._data)
    return raw

def basic_filter(raw,picks):
    raw.notch_filter(50,picks=picks,verbose=False)
    raw.filter(0.1,40,picks=picks,method="iir",verbose=False)
    raw.set_eeg_reference([ch for ch in picks if ch!="VEOG"],verbose=False)
    return raw

def do_ica(raw,picks,use_ica):
    if not use_ica or "VEOG" not in raw.ch_names: return raw
    eeg_picks=[ch for ch in picks if ch!="VEOG"]
    if len(eeg_picks)<3: return raw
    ica=ICA(n_components=min(len(eeg_picks)-1,15),random_state=42,max_iter="auto",verbose=False)
    ica.fit(raw,picks=eeg_picks,verbose=False)
    eog_epochs=create_eog_epochs(raw,ch_name="VEOG",verbose=False)
    inds,_=ica.find_bads_eog(eog_epochs,threshold=3.0)
    if len(inds)>0: ica.exclude=inds; ica.apply(raw)
    return raw

def extract_epochs(raw,t1=9.0,t2=15.0):
    events,event_id=mne.events_from_annotations(raw,verbose=False)
    sf=int(raw.info["sfreq"]); win=sf; length=int(t2-t1)
    eeg_chs=[ch for ch in CHANNELS if ch in raw.ch_names and ch!="VEOG"]
    X=[]; y=[]
    for onset,_,code in events:
        name=[k for k,v in event_id.items() if v==code][0]
        if name in EXCLUDE or name not in VALID_BIN: continue
        start=int(onset+t1*sf); stop=start+length*win
        if stop>raw.n_times: continue
        data=torch.tensor(raw.get_data(picks=eeg_chs,start=start,stop=stop),dtype=torch.float32)
        data=(data-data.mean(-1,keepdim=True))/(data.std(-1,keepdim=True)+1e-6)
        X.append(data.unfold(-1,win,win).permute(1,0,2)); y.append(VALID_BIN[name])
    return torch.stack(X),torch.tensor(y,dtype=torch.long)

def preprocess_subject(vhdr_path,down=256,use_ica=False):
    raw=load_raw(vhdr_path)
    picks=[ch for ch in CHANNELS if ch in raw.ch_names]
    raw=basic_filter(raw,picks)
    raw=do_ica(raw,picks,use_ica)
    raw=raw.resample(down,npad="auto",verbose=False)
    return extract_epochs(raw)

def batch_process(vhdr_paths,use_ica=False):
    X_list=[]
    y_list=[]
    subject_ids=[]
    trial_ids=[]

    global_trial_id=0

    for subj_id,p in enumerate(vhdr_paths):
        Xi,yi=preprocess_subject(p,use_ica=use_ica)
        n_trials=Xi.shape[0]

        X_list.append(Xi)
        y_list.append(yi)

        subject_ids.extend([subj_id]*n_trials)
        trial_ids.extend(range(global_trial_id,global_trial_id+n_trials))
        global_trial_id+=n_trials

    return (
        torch.cat(X_list,dim=0),
        torch.cat(y_list,dim=0),
        torch.tensor(subject_ids),
        torch.tensor(trial_ids)
    )

In [ ]:
prepare_dataset()
vhdrs=sorted(glob(os.path.join(DST_DIR,"*.ahdr")))
X,y,subject_id,trial_id=batch_process(vhdrs,use_ica=USE_ICA)
print("X:", X.shape)  # (N,15,T_samples)
print("y:", y.shape)  # (N,)

/usr/local/lib/python3.12/dist-packages/mne/_fiff/utils.py:76: RuntimeWarning: invalid value encountered in cast
  one = np.asarray(one, dtype=data_view.dtype)


X: torch.Size([560, 6, 15, 256])
y: torch.Size([560])


In [ ]:
class EEG_GAT_Dataset(torch.utils.data.Dataset):
    def __init__(self,X,y):
        self.graphs=[]
        N,S,C,T=X.shape
        for n in range(N):
            for s in range(S):
                x=X[n,s]
                edge_index=torch.combinations(torch.arange(C),r=2).t()
                edge_index=torch.cat([edge_index,edge_index.flip(0)],dim=1)
                self.graphs.append(Data(x=x,edge_index=edge_index,y=y[n]))

    def __len__(self): return len(self.graphs)
    def __getitem__(self,idx): return self.graphs[idx]

In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.query = nn.Conv1d(channels, channels, kernel_size=1)
        self.key = nn.Conv1d(channels, channels, kernel_size=1)
        self.value = nn.Conv1d(channels, channels, kernel_size=1)
        self.scale = channels ** -0.5

    def forward(self, x):
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        attn = torch.softmax(torch.bmm(q.transpose(1,2), k) * self.scale, dim=-1)
        return torch.bmm(v, attn.transpose(1,2))

In [ ]:
class EEG_GAT(nn.Module):
    def __init__(self, num_chans=15, hidden=64, num_classes=2, dropout_rate=0.1):
        super().__init__()

        # ---------- Temporal CNN ----------
        self.temporal1 = nn.Conv2d(1, 30, kernel_size=(1,128), padding=(0,64))
        self.temporal2 = nn.Conv2d(30, 60, kernel_size=(1,64), padding=(0,32))
        self.temporal3 = nn.Conv2d(60, 90, kernel_size=(1,32), padding=(0,16))

        self.bn_t1 = nn.BatchNorm2d(30)
        self.bn_t2 = nn.BatchNorm2d(60)
        self.bn_t3 = nn.BatchNorm2d(90)

        self.prelu = nn.PReLU()
        self.temp_dropout = nn.Dropout2d(0.1)  # ↓ reduced (important)

        # ---------- Spatial (Depthwise over channels) ----------
        self.depthwise = nn.Conv2d(
            90, 90, kernel_size=(num_chans, 1), groups=90, bias=False
        )
        self.bn_s = nn.BatchNorm2d(90)
        self.temp_attn = TemporalAttention(90)

        # ---------- GAT layers ----------
        self.gat1 = GATv2Conv(90, hidden, heads=2, concat=True, dropout=dropout_rate)
        self.gat2 = GATv2Conv(hidden*2, hidden, heads=1, concat=False, dropout=dropout_rate)

        # LayerNorm instead of BatchNorm (very important for EEG)
        self.ln_g1 = nn.LayerNorm(hidden*2)
        self.ln_g2 = nn.LayerNorm(hidden)

        # ---------- Classifier ----------
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 64),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, num_classes)
        )

    def forward(self, data):
        """
        data.x     : (total_nodes, T)
        data.batch : (total_nodes,)
        """
        B = data.batch.max().item() + 1
        C = data.x.size(0) // B
        T = data.x.size(1)

        # ---------- CNN ----------
        x = data.x.view(B, C, T).unsqueeze(1)  # (B,1,C,T)

        x = self.temp_dropout(self.prelu(self.bn_t1(self.temporal1(x))))
        x = self.temp_dropout(self.prelu(self.bn_t2(self.temporal2(x))))
        x = self.temp_dropout(self.prelu(self.bn_t3(self.temporal3(x))))

        x = self.prelu(self.bn_s(self.depthwise(x)))  # (B,90,1,T)
        x = x.squeeze(2)
        x = self.temp_attn(x)              # (B,90,T)

        x = x.mean(dim=-1)        # (B,90)
        x = x.unsqueeze(1).repeat(1, C, 1)  # (B,C,90)
        x = x.reshape(B*C, -1)    # (nodes, features)

        # ---------- GAT ----------
        x = self.prelu(self.ln_g1(self.gat1(x, data.edge_index)))
        x = self.prelu(self.ln_g2(self.gat2(x, data.edge_index)))

        # ---------- Graph Readout ----------
        hG = global_mean_pool(x, data.batch)

        return self.classifier(hG)

In [ ]:
dataset = EEG_GAT_Dataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)
batch = next(iter(loader))
print(batch)

DataBatch(x=[480, 256], edge_index=[2, 6720], y=[32], batch=[480], ptr=[33])


In [ ]:
print(X.shape[2])
print(X.shape[3])

15
256


In [ ]:
print(dataset.graphs[1])

Data(x=[15, 256], edge_index=[2, 210], y=1)


In [ ]:
NUM_EPOCHS = 40
BATCH_SIZE = 32
NUM_FOLDS = 5
PATIENCE = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

graphs = dataset.graphs
kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

root_path = "/content/GAT/KFold"
os.makedirs(root_path, exist_ok=True)

for fold, (train_idx, test_idx) in enumerate(kf.split(graphs)):
    print(f"\n========== Fold {fold+1}/{NUM_FOLDS} ==========")

    fold_path = os.path.join(root_path, f"fold_{fold+1}")
    os.makedirs(fold_path, exist_ok=True)

    train_set = [graphs[i] for i in train_idx]
    test_set = [graphs[i] for i in test_idx]

    print(f"Train samples: {len(train_set)} | Test samples: {len(test_set)}")

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

    model = EEG_GAT(num_chans=X.shape[2], hidden=64, num_classes=2).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
        min_lr=1e-5
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_loss = float("inf")
    patience_counter = 0

    metrics_file = open(os.path.join(fold_path, "metrics.txt"), "w")

    for epoch in range(NUM_EPOCHS):
        model.train()

        train_losses = []
        train_preds = []
        train_labels = []

        for batch in train_loader:
            batch = batch.to(device)

            optimizer.zero_grad()

            out = model(batch)
            loss = criterion(out, batch.y)

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            train_preds.append(out.argmax(dim=1).cpu().numpy())
            train_labels.append(batch.y.cpu().numpy())

        train_preds = np.concatenate(train_preds)
        train_labels = np.concatenate(train_labels)

        train_acc = accuracy_score(train_labels, train_preds) * 100
        train_loss = np.mean(train_losses)

        model.eval()

        test_losses = []
        test_preds = []
        test_labels = []

        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)

                out = model(batch)
                loss = criterion(out, batch.y)

                test_losses.append(loss.item())
                test_preds.append(out.argmax(dim=1).cpu().numpy())
                test_labels.append(batch.y.cpu().numpy())

        test_preds = np.concatenate(test_preds)
        test_labels = np.concatenate(test_labels)

        test_acc = accuracy_score(test_labels, test_preds) * 100
        precision = precision_score(test_labels, test_preds, average="macro", zero_division=0)
        recall = recall_score(test_labels, test_preds, average="macro", zero_division=0)
        f1 = f1_score(test_labels, test_preds, average="macro", zero_division=0)
        kappa = cohen_kappa_score(test_labels, test_preds)

        test_loss = np.mean(test_losses)

        scheduler.step(test_loss)

        print(f"[Fold {fold+1}, Epoch {epoch+1}] Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"                     Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.2f}%")

        metrics_file.write(
            f"epoch={epoch+1}, "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.2f}, "
            f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}, "
            f"precision={precision:.4f}, recall={recall:.4f}, "
            f"f1={f1:.4f}, kappa={kappa:.4f}\n"
        )
        metrics_file.flush()

        if test_loss < best_loss:
            best_loss = test_loss
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(fold_path, "best_model.pt"))
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("Early stopping triggered.")
                break

    metrics_file.close()



========== Fold 1/5 ==========
Train samples: 2136 | Test samples: 534
[Fold 1, Epoch 1] Train Loss: 0.6844 | Train Acc: 57.44%
                     Test  Loss: 0.6809 | Test  Acc: 55.62%
[Fold 1, Epoch 2] Train Loss: 0.6745 | Train Acc: 59.46%
                     Test  Loss: 0.6640 | Test  Acc: 61.61%
[Fold 1, Epoch 3] Train Loss: 0.6610 | Train Acc: 61.75%
                     Test  Loss: 0.6858 | Test  Acc: 60.11%
[Fold 1, Epoch 4] Train Loss: 0.6584 | Train Acc: 62.50%
                     Test  Loss: 0.6666 | Test  Acc: 59.93%
[Fold 1, Epoch 5] Train Loss: 0.6388 | Train Acc: 65.54%
                     Test  Loss: 0.6436 | Test  Acc: 63.11%
[Fold 1, Epoch 6] Train Loss: 0.6262 | Train Acc: 66.29%
                     Test  Loss: 0.6387 | Test  Acc: 65.73%
[Fold 1, Epoch 7] Train Loss: 0.6185 | Train Acc: 68.26%
                     Test  Loss: 0.6348 | Test  Acc: 66.85%
[Fold 1, Epoch 8] Train Loss: 0.6020 | Train Acc: 69.90%
                     Test  Loss: 0.6271 | Test  Acc:

In [ ]:
# ================= CONFIG =================
NUM_EPOCHS = 40
BATCH_SIZE = 32
NUM_FOLDS = 5
PATIENCE = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

graphs = dataset.graphs
kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

root_path = "/content/GAT/KFold"
os.makedirs(root_path, exist_ok=True)

# ================= K-FOLD TRAINING =================
for fold, (train_idx, test_idx) in enumerate(kf.split(graphs)):
    print(f"\n========== Fold {fold+1}/{NUM_FOLDS} ==========")

    fold_path = os.path.join(root_path, f"fold_{fold+1}")
    os.makedirs(fold_path, exist_ok=True)

    train_set = [graphs[i] for i in train_idx]
    test_set  = [graphs[i] for i in test_idx]

    print(f"Train samples: {len(train_set)} | Test samples: {len(test_set)}")

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

    model = EEG_GAT(
        num_chans=X.shape[2],
        hidden=64,
        num_classes=2
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-4,
        weight_decay=1e-3
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
        min_lr=1e-5
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_loss = float("inf")
    patience_counter = 0

    best_test_preds = None
    best_test_labels = None

    metrics_file = open(os.path.join(fold_path, "metrics.txt"), "w")

    # ================= EPOCH LOOP =================
    for epoch in range(NUM_EPOCHS):
        model.train()

        train_losses = []
        train_preds = []
        train_labels = []

        for batch in train_loader:
            batch = batch.to(device)

            optimizer.zero_grad()
            out = model(batch)
            loss = criterion(out, batch.y)

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            train_preds.append(out.argmax(dim=1).cpu().numpy())
            train_labels.append(batch.y.cpu().numpy())

        train_preds  = np.concatenate(train_preds)
        train_labels = np.concatenate(train_labels)

        train_acc  = accuracy_score(train_labels, train_preds) * 100
        train_loss = np.mean(train_losses)

        # ================= EVALUATION =================
        model.eval()

        test_losses = []
        test_preds = []
        test_labels = []

        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)

                out = model(batch)
                loss = criterion(out, batch.y)

                test_losses.append(loss.item())
                test_preds.append(out.argmax(dim=1).cpu().numpy())
                test_labels.append(batch.y.cpu().numpy())

        test_preds  = np.concatenate(test_preds)
        test_labels = np.concatenate(test_labels)

        test_acc  = accuracy_score(test_labels, test_preds) * 100
        precision = precision_score(test_labels, test_preds, average="macro", zero_division=0)
        recall    = recall_score(test_labels, test_preds, average="macro", zero_division=0)
        f1        = f1_score(test_labels, test_preds, average="macro", zero_division=0)
        kappa     = cohen_kappa_score(test_labels, test_preds)

        test_loss = np.mean(test_losses)

        scheduler.step(test_loss)

        print(f"[Fold {fold+1}, Epoch {epoch+1}] "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"                     "
              f"Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.2f}%")

        metrics_file.write(
            f"epoch={epoch+1}, "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.2f}, "
            f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}, "
            f"precision={precision:.4f}, recall={recall:.4f}, "
            f"f1={f1:.4f}, kappa={kappa:.4f}\n"
        )
        metrics_file.flush()

        # ================= EARLY STOPPING =================
        if test_loss < best_loss:
            best_loss = test_loss
            patience_counter = 0

            best_test_preds = test_preds.copy()
            best_test_labels = test_labels.copy()

            torch.save(
                model.state_dict(),
                os.path.join(fold_path, "best_model.pt")
            )
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("Early stopping triggered.")
                break

    metrics_file.close()

    # ================= CONFUSION MATRIX =================
    cm = confusion_matrix(best_test_labels, best_test_preds)

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Class 0", "Class 1"],
        yticklabels=["Class 0", "Class 1"]
    )

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(f"Confusion Matrix - Fold {fold+1}")
    plt.tight_layout()

    plt.savefig(
        os.path.join(fold_path, "confusion_matrix.png"),
        dpi=300
    )
    plt.close()

    np.savetxt(
        os.path.join(fold_path, "confusion_matrix.txt"),
        cm,
        fmt="%d"
    )


========== Fold 1/5 ==========
Train samples: 2688 | Test samples: 672
[Fold 1, Epoch 1] Train Loss: 0.6975 | Train Acc: 51.56%
                     Test  Loss: 0.6931 | Test  Acc: 52.23%
[Fold 1, Epoch 2] Train Loss: 0.6859 | Train Acc: 54.69%
                     Test  Loss: 0.6806 | Test  Acc: 57.29%
[Fold 1, Epoch 3] Train Loss: 0.6778 | Train Acc: 58.56%
                     Test  Loss: 0.6707 | Test  Acc: 59.97%
[Fold 1, Epoch 4] Train Loss: 0.6668 | Train Acc: 60.34%
                     Test  Loss: 0.6701 | Test  Acc: 58.48%
[Fold 1, Epoch 5] Train Loss: 0.6503 | Train Acc: 63.73%
                     Test  Loss: 0.6927 | Test  Acc: 58.04%
[Fold 1, Epoch 6] Train Loss: 0.6444 | Train Acc: 65.03%
                     Test  Loss: 0.6487 | Test  Acc: 65.62%
[Fold 1, Epoch 7] Train Loss: 0.6340 | Train Acc: 64.62%
                     Test  Loss: 0.6141 | Test  Acc: 68.45%
[Fold 1, Epoch 8] Train Loss: 0.6270 | Train Acc: 66.29%
                     Test  Loss: 0.6327 | Test  Acc:

In [ ]:
NUM_EPOCHS = 40
BATCH_SIZE = 32
NUM_FOLDS = 5
PATIENCE = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

graphs = dataset.graphs
kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

root_path = "/content/GAT/KFold"
os.makedirs(root_path, exist_ok=True)

for fold, (train_idx, test_idx) in enumerate(kf.split(graphs)):
    print(f"\n========== Fold {fold+1}/{NUM_FOLDS} ==========")

    fold_path = os.path.join(root_path, f"fold_{fold+1}")
    os.makedirs(fold_path, exist_ok=True)

    train_set = [graphs[i] for i in train_idx]
    test_set = [graphs[i] for i in test_idx]

    print(f"Train samples: {len(train_set)} | Test samples: {len(test_set)}")

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

    model = EEG_GAT(num_chans=X.shape[2], hidden=64, num_classes=2).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
        min_lr=1e-5
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_loss = float("inf")
    patience_counter = 0

    metrics_file = open(os.path.join(fold_path, "metrics.txt"), "w")

    for epoch in range(NUM_EPOCHS):
        model.train()

        train_losses = []
        train_preds = []
        train_labels = []

        for batch in train_loader:
            batch = batch.to(device)

            optimizer.zero_grad()

            out = model(batch)
            loss = criterion(out, batch.y)

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            train_preds.append(out.argmax(dim=1).cpu().numpy())
            train_labels.append(batch.y.cpu().numpy())

        train_preds = np.concatenate(train_preds)
        train_labels = np.concatenate(train_labels)

        train_acc = accuracy_score(train_labels, train_preds) * 100
        train_loss = np.mean(train_losses)

        model.eval()

        test_losses = []
        test_preds = []
        test_labels = []

        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)

                out = model(batch)
                loss = criterion(out, batch.y)

                test_losses.append(loss.item())
                test_preds.append(out.argmax(dim=1).cpu().numpy())
                test_labels.append(batch.y.cpu().numpy())

        test_preds = np.concatenate(test_preds)
        test_labels = np.concatenate(test_labels)

        test_acc = accuracy_score(test_labels, test_preds) * 100
        precision = precision_score(test_labels, test_preds, average="macro", zero_division=0)
        recall = recall_score(test_labels, test_preds, average="macro", zero_division=0)
        f1 = f1_score(test_labels, test_preds, average="macro", zero_division=0)
        kappa = cohen_kappa_score(test_labels, test_preds)

        test_loss = np.mean(test_losses)

        scheduler.step(test_loss)

        print(f"[Fold {fold+1}, Epoch {epoch+1}] Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"                     Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.2f}%")

        metrics_file.write(
            f"epoch={epoch+1}, "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.2f}, "
            f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}, "
            f"precision={precision:.4f}, recall={recall:.4f}, "
            f"f1={f1:.4f}, kappa={kappa:.4f}\n"
        )
        metrics_file.flush()

        if test_loss < best_loss:
            best_loss = test_loss
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(fold_path, "best_model.pt"))
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("Early stopping triggered.")
                break

    metrics_file.close()


========== Fold 1/5 ==========
Train samples: 2688 | Test samples: 672
[Fold 1, Epoch 1] Train Loss: 0.7001 | Train Acc: 51.82%
                     Test  Loss: 0.6890 | Test  Acc: 54.76%
[Fold 1, Epoch 2] Train Loss: 0.6904 | Train Acc: 54.39%
                     Test  Loss: 0.6839 | Test  Acc: 55.06%
[Fold 1, Epoch 3] Train Loss: 0.6769 | Train Acc: 58.71%
                     Test  Loss: 0.6718 | Test  Acc: 58.18%
[Fold 1, Epoch 4] Train Loss: 0.6726 | Train Acc: 59.67%
                     Test  Loss: 0.6640 | Test  Acc: 61.31%
[Fold 1, Epoch 5] Train Loss: 0.6655 | Train Acc: 61.42%
                     Test  Loss: 0.6541 | Test  Acc: 62.65%
[Fold 1, Epoch 6] Train Loss: 0.6536 | Train Acc: 62.65%
                     Test  Loss: 0.6577 | Test  Acc: 61.31%
[Fold 1, Epoch 7] Train Loss: 0.6288 | Train Acc: 66.74%
                     Test  Loss: 0.6437 | Test  Acc: 66.67%
[Fold 1, Epoch 8] Train Loss: 0.6225 | Train Acc: 67.34%
                     Test  Loss: 0.6643 | Test  Acc:

In [ ]:
import os
import shutil
import zipfile
from google.colab import files

def save_file():
    base_path = "/content/GAT/KFold"
    tmp_dir   = "/content/tmp_download"
    zip_path  = "/content/EEG-GAT_v2_Trial_All_Folds.zip"

    # clean temp dir
    if os.path.exists(tmp_dir):
        shutil.rmtree(tmp_dir)
    os.makedirs(tmp_dir, exist_ok=True)

    # copy + rename
    for fold in sorted(os.listdir(base_path)):
        fold_path = os.path.join(base_path, fold)
        if not os.path.isdir(fold_path):
            continue

        for fname in os.listdir(fold_path):
            src = os.path.join(fold_path, fname)
            if os.path.isfile(src):
                new_name = f"{fold}_{fname}"
                shutil.copy2(src, os.path.join(tmp_dir, new_name))

    print("Files copied.")

    # zip
    if os.path.exists(zip_path):
        os.remove(zip_path)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for f in os.listdir(tmp_dir):
            zipf.write(os.path.join(tmp_dir, f), f)

    print(f"ZIP created: {zip_path}")
    files.download(zip_path)

In [ ]:
save_file()

Files copied.
ZIP created: /content/EEG-GAT_v2_Trial_All_Folds.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Trainings

In [ ]:
NUM_EPOCHS = 40
BATCH_SIZE = 32
NUM_FOLDS = 5
PATIENCE = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

graphs = dataset.graphs
kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

root_path = "/content/GAT/KFold"
os.makedirs(root_path, exist_ok=True)

for fold, (train_idx, test_idx) in enumerate(kf.split(graphs)):
    print(f"\n========== Fold {fold+1}/{NUM_FOLDS} ==========")

    fold_path = os.path.join(root_path, f"fold_{fold+1}")
    os.makedirs(fold_path, exist_ok=True)

    train_set = [graphs[i] for i in train_idx]
    test_set = [graphs[i] for i in test_idx]

    print(f"Train samples: {len(train_set)} | Test samples: {len(test_set)}")

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

    model = EEG_GAT(num_chans=X.shape[2], hidden=128, num_classes=2).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
        min_lr=1e-5
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_loss = float("inf")
    patience_counter = 0

    metrics_file = open(os.path.join(fold_path, "metrics.txt"), "w")

    for epoch in range(NUM_EPOCHS):
        model.train()

        train_losses = []
        train_preds = []
        train_labels = []

        for batch in train_loader:
            batch = batch.to(device)

            optimizer.zero_grad()

            out = model(batch)
            loss = criterion(out, batch.y)

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            train_preds.append(out.argmax(dim=1).cpu().numpy())
            train_labels.append(batch.y.cpu().numpy())

        train_preds = np.concatenate(train_preds)
        train_labels = np.concatenate(train_labels)

        train_acc = accuracy_score(train_labels, train_preds) * 100
        train_loss = np.mean(train_losses)

        model.eval()

        test_losses = []
        test_preds = []
        test_labels = []

        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)

                out = model(batch)
                loss = criterion(out, batch.y)

                test_losses.append(loss.item())
                test_preds.append(out.argmax(dim=1).cpu().numpy())
                test_labels.append(batch.y.cpu().numpy())

        test_preds = np.concatenate(test_preds)
        test_labels = np.concatenate(test_labels)

        test_acc = accuracy_score(test_labels, test_preds) * 100
        precision = precision_score(test_labels, test_preds, average="macro", zero_division=0)
        recall = recall_score(test_labels, test_preds, average="macro", zero_division=0)
        f1 = f1_score(test_labels, test_preds, average="macro", zero_division=0)
        kappa = cohen_kappa_score(test_labels, test_preds)

        test_loss = np.mean(test_losses)

        scheduler.step(test_loss)

        print(f"[Fold {fold+1}, Epoch {epoch+1}] Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"                     Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.2f}%")

        metrics_file.write(
            f"epoch={epoch+1}, "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.2f}, "
            f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}, "
            f"precision={precision:.4f}, recall={recall:.4f}, "
            f"f1={f1:.4f}, kappa={kappa:.4f}\n"
        )
        metrics_file.flush()

        if test_loss < best_loss:
            best_loss = test_loss
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(fold_path, "best_model.pt"))
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("Early stopping triggered.")
                break

    metrics_file.close()


========== Fold 1/5 ==========
Train samples: 2688 | Test samples: 672
[Fold 1, Epoch 1] Train Loss: 0.6968 | Train Acc: 52.90%
                     Test  Loss: 0.6928 | Test  Acc: 52.98%
[Fold 1, Epoch 2] Train Loss: 0.6885 | Train Acc: 54.20%
                     Test  Loss: 0.6844 | Test  Acc: 56.40%
[Fold 1, Epoch 3] Train Loss: 0.6848 | Train Acc: 57.81%
                     Test  Loss: 0.6801 | Test  Acc: 56.55%
[Fold 1, Epoch 4] Train Loss: 0.6774 | Train Acc: 58.44%
                     Test  Loss: 0.6713 | Test  Acc: 58.93%
[Fold 1, Epoch 5] Train Loss: 0.6714 | Train Acc: 59.11%
                     Test  Loss: 0.6659 | Test  Acc: 62.35%
[Fold 1, Epoch 6] Train Loss: 0.6575 | Train Acc: 62.39%
                     Test  Loss: 0.6629 | Test  Acc: 61.90%
[Fold 1, Epoch 7] Train Loss: 0.6463 | Train Acc: 64.92%
                     Test  Loss: 0.6458 | Test  Acc: 64.29%
[Fold 1, Epoch 8] Train Loss: 0.6358 | Train Acc: 65.14%
                     Test  Loss: 0.6391 | Test  Acc:

In [ ]:
NUM_EPOCHS = 40
BATCH_SIZE = 32
NUM_FOLDS = 5
PATIENCE = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

graphs = dataset.graphs
graphs_per_trial = X.shape[1]
graph_subject_ids = subject_id.repeat_interleave(graphs_per_trial)

gkf = GroupKFold(n_splits=NUM_FOLDS)
#kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

root_path = "/content/GAT/KFold"
os.makedirs(root_path, exist_ok=True)

for fold, (train_idx, test_idx) in enumerate(gkf.split(graphs, y=None, groups=graph_subject_ids)):
    print(f"\n========== Fold {fold+1}/{NUM_FOLDS} ==========")

    fold_path = os.path.join(root_path, f"fold_{fold+1}")
    os.makedirs(fold_path, exist_ok=True)

    train_set = [graphs[i] for i in train_idx]
    test_set = [graphs[i] for i in test_idx]

    print(f"Train samples: {len(train_set)} | Test samples: {len(test_set)}")

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

    model = EEG_GAT(num_chans=X.shape[2], hidden=128, num_classes=2).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
        min_lr=1e-5
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_loss = float("inf")
    patience_counter = 0

    metrics_file = open(os.path.join(fold_path, "metrics.txt"), "w")

    for epoch in range(NUM_EPOCHS):
        model.train()

        train_losses = []
        train_preds = []
        train_labels = []

        for batch in train_loader:
            batch = batch.to(device)

            optimizer.zero_grad()

            out = model(batch)
            loss = criterion(out, batch.y)

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            train_preds.append(out.argmax(dim=1).cpu().numpy())
            train_labels.append(batch.y.cpu().numpy())

        train_preds = np.concatenate(train_preds)
        train_labels = np.concatenate(train_labels)

        train_acc = accuracy_score(train_labels, train_preds) * 100
        train_loss = np.mean(train_losses)

        model.eval()

        test_losses = []
        test_preds = []
        test_labels = []

        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)

                out = model(batch)
                loss = criterion(out, batch.y)

                test_losses.append(loss.item())
                test_preds.append(out.argmax(dim=1).cpu().numpy())
                test_labels.append(batch.y.cpu().numpy())

        test_preds = np.concatenate(test_preds)
        test_labels = np.concatenate(test_labels)

        test_acc = accuracy_score(test_labels, test_preds) * 100
        precision = precision_score(test_labels, test_preds, average="macro", zero_division=0)
        recall = recall_score(test_labels, test_preds, average="macro", zero_division=0)
        f1 = f1_score(test_labels, test_preds, average="macro", zero_division=0)
        kappa = cohen_kappa_score(test_labels, test_preds)

        test_loss = np.mean(test_losses)

        scheduler.step(test_loss)

        print(f"[Fold {fold+1}, Epoch {epoch+1}] Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"                     Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.2f}%")

        metrics_file.write(
            f"epoch={epoch+1}, "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.2f}, "
            f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}, "
            f"precision={precision:.4f}, recall={recall:.4f}, "
            f"f1={f1:.4f}, kappa={kappa:.4f}\n"
        )
        metrics_file.flush()

        if test_loss < best_loss:
            best_loss = test_loss
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(fold_path, "best_model.pt"))
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("Early stopping triggered.")
                break

    metrics_file.close()


========== Fold 1/5 ==========
Train samples: 2754 | Test samples: 606
[Fold 1, Epoch 1] Train Loss: 0.6927 | Train Acc: 53.27%
                     Test  Loss: 0.7251 | Test  Acc: 42.41%
[Fold 1, Epoch 2] Train Loss: 0.6728 | Train Acc: 59.08%
                     Test  Loss: 0.7485 | Test  Acc: 45.21%
[Fold 1, Epoch 3] Train Loss: 0.6608 | Train Acc: 61.18%
                     Test  Loss: 0.7741 | Test  Acc: 40.76%
[Fold 1, Epoch 4] Train Loss: 0.6400 | Train Acc: 64.67%
                     Test  Loss: 0.7959 | Test  Acc: 45.21%
[Fold 1, Epoch 5] Train Loss: 0.6271 | Train Acc: 66.63%
                     Test  Loss: 0.8615 | Test  Acc: 39.27%
[Fold 1, Epoch 6] Train Loss: 0.6064 | Train Acc: 70.04%
                     Test  Loss: 0.8044 | Test  Acc: 44.88%
[Fold 1, Epoch 7] Train Loss: 0.5921 | Train Acc: 69.90%
                     Test  Loss: 0.8564 | Test  Acc: 44.88%
[Fold 1, Epoch 8] Train Loss: 0.5714 | Train Acc: 72.98%
                     Test  Loss: 0.9015 | Test  Acc:

In [ ]:
NUM_EPOCHS = 40
BATCH_SIZE = 32
NUM_FOLDS = 5
PATIENCE = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
graph_trial_ids = []

for n in range(X.shape[0]):          # trials
    for s in range(X.shape[1]):      # windows per trial
        graph_trial_ids.append(trial_id[n].item())

graph_trial_ids = np.array(graph_trial_ids)

gkf = GroupKFold(n_splits=NUM_FOLDS)
#kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

root_path = "/content/GAT/KFold"
os.makedirs(root_path, exist_ok=True)

for fold, (train_idx, test_idx) in enumerate(gkf.split(graphs, y=None, groups=graph_trial_ids)):
    print(f"\n========== Fold {fold+1}/{NUM_FOLDS} ==========")

    fold_path = os.path.join(root_path, f"fold_{fold+1}")
    os.makedirs(fold_path, exist_ok=True)

    train_set = [graphs[i] for i in train_idx]
    test_set = [graphs[i] for i in test_idx]

    print(f"Train samples: {len(train_set)} | Test samples: {len(test_set)}")

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

    model = EEG_GAT(num_chans=X.shape[2], hidden=128, num_classes=2).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
        min_lr=1e-5
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_loss = float("inf")
    patience_counter = 0

    metrics_file = open(os.path.join(fold_path, "metrics.txt"), "w")

    for epoch in range(NUM_EPOCHS):
        model.train()

        train_losses = []
        train_preds = []
        train_labels = []

        for batch in train_loader:
            batch = batch.to(device)

            optimizer.zero_grad()

            out = model(batch)
            loss = criterion(out, batch.y)

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            train_preds.append(out.argmax(dim=1).cpu().numpy())
            train_labels.append(batch.y.cpu().numpy())

        train_preds = np.concatenate(train_preds)
        train_labels = np.concatenate(train_labels)

        train_acc = accuracy_score(train_labels, train_preds) * 100
        train_loss = np.mean(train_losses)

        model.eval()

        test_losses = []
        test_preds = []
        test_labels = []

        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)

                out = model(batch)
                loss = criterion(out, batch.y)

                test_losses.append(loss.item())
                test_preds.append(out.argmax(dim=1).cpu().numpy())
                test_labels.append(batch.y.cpu().numpy())

        test_preds = np.concatenate(test_preds)
        test_labels = np.concatenate(test_labels)

        test_acc = accuracy_score(test_labels, test_preds) * 100
        precision = precision_score(test_labels, test_preds, average="macro", zero_division=0)
        recall = recall_score(test_labels, test_preds, average="macro", zero_division=0)
        f1 = f1_score(test_labels, test_preds, average="macro", zero_division=0)
        kappa = cohen_kappa_score(test_labels, test_preds)

        test_loss = np.mean(test_losses)

        scheduler.step(test_loss)

        print(f"[Fold {fold+1}, Epoch {epoch+1}] Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"                     Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.2f}%")

        metrics_file.write(
            f"epoch={epoch+1}, "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.2f}, "
            f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}, "
            f"precision={precision:.4f}, recall={recall:.4f}, "
            f"f1={f1:.4f}, kappa={kappa:.4f}\n"
        )
        metrics_file.flush()

        if test_loss < best_loss:
            best_loss = test_loss
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(fold_path, "best_model.pt"))
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("Early stopping triggered.")
                break

    metrics_file.close()


========== Fold 1/5 ==========
Train samples: 2688 | Test samples: 672
[Fold 1, Epoch 1] Train Loss: 0.6983 | Train Acc: 52.12%
                     Test  Loss: 0.6908 | Test  Acc: 55.06%
[Fold 1, Epoch 2] Train Loss: 0.6861 | Train Acc: 54.95%
                     Test  Loss: 0.6891 | Test  Acc: 55.80%
[Fold 1, Epoch 3] Train Loss: 0.6841 | Train Acc: 56.36%
                     Test  Loss: 0.6804 | Test  Acc: 56.25%
[Fold 1, Epoch 4] Train Loss: 0.6823 | Train Acc: 56.58%
                     Test  Loss: 0.6735 | Test  Acc: 58.78%
[Fold 1, Epoch 5] Train Loss: 0.6694 | Train Acc: 60.79%
                     Test  Loss: 0.6610 | Test  Acc: 62.05%
[Fold 1, Epoch 6] Train Loss: 0.6649 | Train Acc: 61.38%
                     Test  Loss: 0.6697 | Test  Acc: 58.48%
[Fold 1, Epoch 7] Train Loss: 0.6503 | Train Acc: 63.65%
                     Test  Loss: 0.6248 | Test  Acc: 68.75%
[Fold 1, Epoch 8] Train Loss: 0.6394 | Train Acc: 65.40%
                     Test  Loss: 0.6325 | Test  Acc:

In [ ]:
import os
import shutil
import zipfile
from google.colab import files

def save_file():
    base_path = "/content/GAT/KFold"
    tmp_dir   = "/content/tmp_download"
    zip_path  = "/content/GATv2_2Layers_TrialGrpKFold_All_Folds.zip"

    # clean temp dir
    if os.path.exists(tmp_dir):
        shutil.rmtree(tmp_dir)
    os.makedirs(tmp_dir, exist_ok=True)

    # copy + rename
    for fold in sorted(os.listdir(base_path)):
        fold_path = os.path.join(base_path, fold)
        if not os.path.isdir(fold_path):
            continue

        for fname in os.listdir(fold_path):
            src = os.path.join(fold_path, fname)
            if os.path.isfile(src):
                new_name = f"{fold}_{fname}"
                shutil.copy2(src, os.path.join(tmp_dir, new_name))

    print("Files copied.")

    # zip
    if os.path.exists(zip_path):
        os.remove(zip_path)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for f in os.listdir(tmp_dir):
            zipf.write(os.path.join(tmp_dir, f), f)

    print(f"ZIP created: {zip_path}")
    files.download(zip_path)

In [ ]:
save_file()

Files copied.
ZIP created: /content/GATv2_2Layers_TrialGrpKFold_All_Folds.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>